# BibleAssistant

<a target="_blank" href="https://colab.research.google.com/github/simonguest/CS-394/blob/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/simonguest/CS-394/raw/refs/heads/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Data generation settings

In [11]:
from pydantic import BaseModel
from typing import List, Literal, Optional, Dict, Any, Tuple
from pydantic import BaseModel, Field, ValidationError

NUM_TRAIN_EXAMPLES = 2000  # @param {type:"number"}
NUM_VAL_EXAMPLES = 200  # @param {type:"number"}
NUM_TEST_EXAMPLES = 20 # @param {type:"number"}
TEMPERATURE = 0.8  # @param {type:"number"}

DATA_FOLDER = "./.data/generated"
!mkdir -p {DATA_FOLDER}

DATAGEN_MODEL = "openai/gpt-5-nano"

## Dataset diversity

In [20]:
BIBLE_BOOKS = [
    "Old Testament",
    "New Testament",
]

TOPIC_CATEGORIES = [
    "anxiety", "fear", "grief", "sadness", "anger", "loneliness", "jealousy", "shame", "guilt", "regret", "hopelessness",
    "confusion", "overthinking", "self_doubt", "burnout", "emptiness", "lack_of_purpose", "identity_crisis", "faith_doubt", "spiritual_dryness", "temptation", "sin", "repentance", "forgiveness", "obedience",
    "conflict", "rejection", "betrayal", "trust_issues", "family_issues", "friendship_issues", "love", "breakup",
    "failure", "success", "uncertainty", "decision_making", "change", "waiting", "loss", "pressure",
    "hope", "healing", "gratitude", "purpose", "motivation", "discipline", "resilience", "self_improvement",
    "meaning_of_life", "suffering", "death", "afterlife", "justice", "free_will", "question",
    "awakening", "clarity", "questioning_beliefs", "critical_thinking", "self_trust", "inner_strength", "truth_seeking", "breaking_control", "emotional_release", "identity_reclaiming", "freedom", "empowerment",
]

## Model for structured output

In [21]:
class BibleExplanation(BaseModel):
    verse: str
    explanation: str

class BibleConversation(BaseModel):
    user: str
    assistant: BibleExplanation


## Get OpenRouter API key

In [22]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OpenRouter')
else:
  load_dotenv()

## Conversation generation functions

In [23]:
import openai
import os

client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def generate_completion(prompt: str) -> Optional[BibleConversation]:
    try:
        response = client.responses.parse(
            model=DATAGEN_MODEL,
            input=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            stream=False,
            text_format=BibleConversation,
        )
        return response.output_parsed
    except Exception:
        return None

def create_conversation(topic: str) -> Optional[BibleConversation]:
    prompt = f"""
    You are a playful bible assistant bot. Answer the question about {topic} using bible verese

    Rules:
    - Use ONLY books from this list:
    {", ".join(BIBLE_BOOKS)}
    - Do NOT cite any book outside the allowed list

    Return the following:
    1. The single Bible selected verse and its content as a single string.
    2. One paragraphs in one string explaining how the verses address the {topic}.
    """

    return generate_completion(prompt)

## Dataset generation functions

In [24]:
import random
import json
from tqdm import tqdm
import os

def generate_dataset_files(num_examples: int, jsonl_path: str) -> None:
    os.makedirs(os.path.dirname(jsonl_path), exist_ok=True)

    with open(jsonl_path, "w", encoding="utf-8") as f_jsonl:
        for idx in tqdm(range(num_examples)):
            topic = random.choice(TOPIC_CATEGORIES)

            conv = None
            tries = 0
            while conv is None and tries < 6:
                conv = create_conversation(topic)
                tries += 1

            if conv is None:
                print(f"Error generating conversation for example {idx}")
                continue

            template = {
                "messages": [
                    {"role": "user", "content": conv.user},
                    {
                        "role": "assistant",
                        "content": json.dumps(conv.assistant.model_dump(), ensure_ascii=False),
                    },
                ]
            }

            f_jsonl.write(json.dumps(template, ensure_ascii=False) + "\n")

## Generate all the data!

In [ ]:
from datetime import datetime

TRAIN_FILE = f"{DATA_FOLDER}/train_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"

generate_dataset_files(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
generate_dataset_files(NUM_VAL_EXAMPLES, VALID_FILE)
generate_dataset_files(NUM_TEST_EXAMPLES, TEST_FILE)

  0%|          | 3/2000 [01:31<16:37:11, 29.96s/it]

# What use case you chose and why?
In this world, people suffer from many issues in their lives. Therefore, I would like to create a Bible assistant that can answer people's questions about the Bible and help them feel better. I believe the Bible has the ability to heal everyone in the world. The reason I want to create this Bible AI assistant is that it can help answer the questions in people's hearts right away, without making them wait.

# Your diversity dimensions and how they shape the dataset

For the diversity dimension, I limited the Bible books to the Old and New Testaments because they are widely accepted Christian scriptures around the world. I want to ensure that users are protected from false teachings or misleading interpretations that may come from cult sources on the internet.

Regarding input, I expect users to either ask a question about the Bible or share a personal issue they are facing. For the output, the system should first restate the user’s concern in its own words to show understanding. Then it should select an appropriate verse, explain its meaning in clear and simple language—since many verses can be difficult to understand—and provide practical guidance on what the user can do. Finally, it should include a short compassionate note to make the response feel warm and caring.

With this diversity dimension, users will feel as though they are consulting with a sincere and thoughtful Christian.

# Observations from your quality review (what looked good, what needed fixing)

# Sample

{"messages":
[
  {"role": "user", "content": "I've been hurt by someone who betrayed my trust, and I'm trying to forgive and move forward. I'm seeking guidance and peace rather than staying in anger."},
  
  {"role": "assistant", "content": "{
    \"issue_question\": \"I've been hurt by someone who betrayed my trust, and I'm trying to forgive and move forward. I'm seeking guidance and peace rather than staying in anger.\",

   \"verse\": \"Matthew 6:14-15 - For if ye forgive men their trespasses, your heavenly Father will forgive you: But if ye forgive not men their trespasses, neither will your Father forgive your trespasses.\",
   
   \"explanation\": \"These verses teach that forgiveness is a choice we make, not merely a feeling that comes and goes. By choosing to forgive, we align with God’s mercy and open ourselves to healing, even when hurt remains. The passage also cautions that withholding forgiveness can hinder our own experience of divine mercy, reminding us that mercy towards others is connected to mercy received.\",
   
   \"guidance\": [\"Take a moment to acknowledge the hurt and name what you want to release.\", \"Ask God for the strength to forgive and release bitterness.\", \"Choose to forgive in your heart today, even if emotions linger.\", \"If safe and possible, speak with the person about forgiveness in a calm, boundaries-respecting way.\", \"Journal or pray daily about your progress and seek support from a trusted friend or counselor.\"],
   
   \"note\": \"Be gentle with yourself—forgiveness is a process, and progress may be gradual.\"}"}
   ]
   }

# What Looked Good?

I think it looks good so far, it actually feels like what I would expect when consulting with a Christian

# What Needed Fixing?

Having the agent stating the issue feel unnecessary. It doesn't speak in their own word as I expected, but it's fine

# Lesson Learned

I initially tried training a fighting game helper model, but it generated unusable data. I think this happened because fighting games are a niche topic and require highly specific knowledge. Most of that knowledge is found in YouTube tutorial videos rather than structured text sources.

In contrast, the Bible is a widely known and well-documented topic with abundant written material available. That is likely why this approach works much better for the Bible-based assistant.